# AutoEvolve: Classifying S12/S21 vs. Cooper s7/s10

This notebook uses machine learning to classify Picard-Fuchs ODEs from your discoveries.json.

## Goals
- Train a classifier to distinguish Order-2 (S12/S21) vs. Order-3 (Cooper s7/s10) sequences.
- Validate against your empirical data (K3-DISC-0003, K3-DISC-0035, etc.).
- Identify topological signatures (Betti numbers, Euler χ, Wasserstein distance).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load your discoveries
try:
    df = pd.read_json('../data/discoveries.json')
except FileNotFoundError:
    # Fallback to sample data if discoveries.json is not found
    data = {
        'id': ['K3-DISC-0003', 'K3-DISC-0025', 'K3-DISC-0005', 'K3-DISC-0035'],
        's12': [1.1, 1.05, 1.15, 1.177],
        's21': [0.9, 0.95, 0.85, 0.849],
        'max_asymmetry': [47.0, 41.0, 38.0, 33.0],
        'euler_characteristic': [2, 7, -3, 7],
        'family': ['GALAXY_GROUP', 'SATELLITE_SYSTEM', 'GALAXY_GROUP', 'SATELLITE_SYSTEM']
    }
    df = pd.DataFrame(data)

# Display the first few rows
df.head()

In [ ]:
# Prepare features and labels
X = df[['s12', 's21', 'max_asymmetry', 'euler_characteristic']]
# Map family to binary labels: 0 for SATELLITE_SYSTEM (S12/S21), 1 for GALAXY_GROUP (Cooper)
y = df['family'].apply(lambda x: 0 if x == 'SATELLITE_SYSTEM' else 1)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display shapes
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')

In [ ]:
# Train a Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

# Classification report
print(classification_report(y_test, y_pred, target_names=['SATELLITE_SYSTEM', 'GALAXY_GROUP']))

In [ ]:
# Feature importance
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': clf.feature_importances_
})
feature_importances = feature_importances.sort_values('Importance', ascending=False)

# Plot feature importances
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importances)
plt.title('Feature Importance for Picard-Fuchs ODE Classification')
plt.show()

## Next Steps

1. **Validate with your full `discoveries.json` dataset.**
2. **Compare with Cooper s7/s10 sequences** (Order-3 ODEs).
3. **Integrate with Lean 4 formalization** (e.g., `FTheoryFibration.lean`).
4. **Extend to other topological metrics** (Betti numbers, Wasserstein distance).